# NB07 — Advanced Evidence Integration
**Goal:** Repair and extend the producer evidence matrix from 5 broken streams to 7 working streams.

| Priority | Analysis | Output |
|---|---|---|
| P1 | Within-stage Spearman correlations (Stage_I_II / Stage_III_IV) | `within_Stage_*.csv`, `within_stage_both_significant.csv` |
| P2 | Fix GLASSO pipeline (alpha sweep, lower threshold) | `d_glasso_partial_correlation_edges_fixed.csv` |
| P3 | Genus-level enzyme cross-linking | Updated `E3_kegg` |
| P4 | Stage-stratified ML on Advanced CRC (RF + SHAP) | `advanced_crc_ml_shap_results.csv` |
| P5 | Rebuild evidence matrix (7 streams, correct column names) | `producer_evidence_matrix_v2.csv` |

In [1]:
import sys, warnings, pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# ── paths ──────────────────────────────────────────────────────────────────────
NB_DIR    = Path(".").resolve()                              # publication/
INTER_DIR = Path(r"C:\Users\andna\Documents\KI\Results\intermediate")
TABLE_DIR = Path(r"C:\Users\andna\Documents\KI\Results\tables")

sys.path.insert(0, str(NB_DIR))
from utils import (
    DATASET_PRIMARY,
    FDR_THRESHOLD, MIN_CORR,
    load_pickle,
    spearman_correlation_matrix,
    extract_genus,
)

# ── load preprocessed data ────────────────────────────────────────────────────
pkl        = load_pickle(INTER_DIR / "preprocessed_data.pkl")
species_df = pkl["species_clr"][DATASET_PRIMARY].copy()     # (347, 4392)
mtb_df     = pkl["mtb_log10"][DATASET_PRIMARY].copy()       # (347, 249)
meta_df    = pkl["harmonized_meta"][DATASET_PRIMARY].copy() # (347, 18)

print(f"Species:     {species_df.shape}")
print(f"Metabolites: {mtb_df.shape}")
print(f"Metadata:    {meta_df.shape}")
print()
print("Stage.6 counts:")
print(meta_df["Stage.6"].value_counts())

# ── polyamine KEGG IDs and EC mapping ─────────────────────────────────────────
POLYAMINE_KEGG = {"C00134", "C00315", "C00378", "C00986", "C01672", "C00077", "C00062"}

POLYAMINE_EC = {
    "C00134": ["4.1.1.17", "4.1.1.18", "4.1.1.19"],  # Putrescine: ODC, LDC, ADC
    "C00315": ["2.5.1.16"],                             # Spermidine synthase
    "C00378": ["2.5.1.22"],                             # Spermine synthase
    "C00986": ["4.1.1.19", "3.5.3.11"],                # Agmatine: ADC, agmatinase
    "C01672": ["4.1.1.18"],                             # Cadaverine: LDC
    "C00077": ["4.1.1.17"],                             # Ornithine: ODC
    "C00062": ["4.1.1.19", "3.5.3.11"],                # Arginine: ADC, agmatinase
}

# Polyamine metabolite columns present in mtb_df
poly_cols = [c for c in mtb_df.columns if c.split("_")[0] in POLYAMINE_KEGG]
print(f"\nPolyamine columns in mtb_df ({len(poly_cols)}): {poly_cols}")

Loaded: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl
Species:     (347, 4392)
Metabolites: (347, 249)
Metadata:    (347, 18)

Stage.6 counts:
Stage.6
Healthy         127
Stage_I_II       69
Stage_III_IV     54
MP               40
HS               30
Stage_0          27
Name: count, dtype: int64

Polyamine columns in mtb_df (7): ['C00062_Arg', 'C01672_Cadaverine', 'C00077_Ornithine', 'C00134_Putrescine', 'C00315_Spermidine', 'C00378_Thiamine', 'C00986_1,3-Diaminopropane']


## Priority 1 — Within-Stage Spearman Correlations
Compute species × metabolite Spearman correlations *within* Stage_I_II and Stage_III_IV separately.  
Pairs significant (FDR < 0.05) in **both** stages are strongest producer candidates, independent of between-stage confounding.

In [2]:
stage_corr = {}

for stage in ["Stage_I_II", "Stage_III_IV"]:
    idx = meta_df[meta_df["Stage.6"] == stage].index
    spe_s = species_df.loc[idx]
    mtb_s = mtb_df.loc[idx]
    print(f"{stage}: {len(idx)} samples")

    # Restrict to top-variance species (speeds up; mirrors NB02 MAX_SPECIES_NB02=500)
    top_sp = spe_s.var(axis=0).nlargest(500).index
    spe_s  = spe_s[top_sp]

    corr = spearman_correlation_matrix(spe_s, mtb_s, fdr=FDR_THRESHOLD, min_r=0.0)
    corr["stage"] = stage
    stage_corr[stage] = corr
    corr.to_csv(TABLE_DIR / f"within_{stage}_correlations.csv", index=False)
    print(f"  Total pairs: {len(corr):,}  |  FDR<0.10: {(corr.qval < 0.10).sum():,}")

sig_s1 = stage_corr["Stage_I_II"][stage_corr["Stage_I_II"]["qval"] < 0.10][["species","metabolite","rho","qval"]].rename(columns={"rho":"rho_s1","qval":"qval_s1"})
sig_s2 = stage_corr["Stage_III_IV"][stage_corr["Stage_III_IV"]["qval"] < 0.10][["species","metabolite","rho","qval"]].rename(columns={"rho":"rho_s2","qval":"qval_s2"})

# ── Intersection (original) ──────────────────────────────────────────────────
both_sig = pd.merge(sig_s1, sig_s2, on=["species","metabolite"])
both_sig["same_direction"] = np.sign(both_sig["rho_s1"]) == np.sign(both_sig["rho_s2"])
both_sig = both_sig.sort_values("rho_s1", key=abs, ascending=False).reset_index(drop=True)
both_sig.to_csv(TABLE_DIR / "within_stage_both_significant.csv", index=False)

# ── Union (either stage sig; used for E7 evidence stream) ────────────────────
either_sig = pd.merge(sig_s1, sig_s2, on=["species","metabolite"], how="outer")
either_sig["rho_s1"] = either_sig["rho_s1"].fillna(0.0)
either_sig["rho_s2"] = either_sig["rho_s2"].fillna(0.0)
either_sig["same_direction"] = (
    (either_sig["rho_s1"] == 0.0) |   # only sig in s2
    (either_sig["rho_s2"] == 0.0) |   # only sig in s1
    (np.sign(either_sig["rho_s1"]) == np.sign(either_sig["rho_s2"]))  # both sig, same dir
)
either_sig["max_rho"] = either_sig[["rho_s1","rho_s2"]].abs().max(axis=1)
either_sig = (either_sig[either_sig["same_direction"]]
              .sort_values("max_rho", ascending=False)
              .reset_index(drop=True))
either_sig.to_csv(TABLE_DIR / "within_stage_either_significant.csv", index=False)

print(f"\nPairs significant in BOTH stages: {len(both_sig):,}")
print(f"Pairs significant in EITHER stage (consistent direction): {len(either_sig):,}")
print(f"  Stage_I_II only: {(either_sig['qval_s1'].isna()).sum():,}")
print(f"  Stage_III_IV only: {(either_sig['qval_s2'].isna()).sum():,}")
print(f"  Both: {(either_sig['qval_s1'].notna() & either_sig['qval_s2'].notna()).sum():,}")
print("\nTop 15 either-sig pairs:")
print(either_sig.sort_values("max_rho", ascending=False).head(15)[
    ["species","metabolite","rho_s1","qval_s1","rho_s2","qval_s2"]
].to_string(index=False))


Stage_I_II: 69 samples
  Total pairs: 34  |  FDR<0.10: 34
Stage_III_IV: 54 samples
  Total pairs: 81  |  FDR<0.10: 81

Pairs significant in BOTH stages: 1
Pairs significant in EITHER stage (consistent direction): 114
  Stage_I_II only: 80
  Stage_III_IV only: 33
  Both: 1

Top 15 either-sig pairs:
                       species                metabolite    rho_s1  qval_s1    rho_s2  qval_s2
            CAG-83 sp001916855  C00624_N-Acetylglutamate  0.000000      NaN  0.663274 0.005752
               ER4 sp900552015                C00055_CMP  0.000000      NaN  0.626999 0.023746
           UBA1417 sp003531055 C02714_N-Acetylputrescine  0.000000      NaN -0.615476 0.023746
         Avimicrobium caecorum  C00624_N-Acetylglutamate  0.000000      NaN  0.612426 0.023746
Faecalibacterium prausnitzii_C         C00398_Tryptamine -0.604792 0.004632  0.000000      NaN
         Faecousia sp003525905 C02714_N-Acetylputrescine  0.000000      NaN -0.604650 0.023746
            CAG-83 sp900545585      

## Priority 2 — Fix GLASSO Pipeline
Previous run produced an empty output file. Root cause: over-regularisation + threshold too strict.  
Fix: sweep alpha with `GraphicalLassoCV`, compute partial correlations from precision matrix, use threshold 0.05.

In [3]:
from sklearn.covariance import GraphicalLassoCV
from sklearn.preprocessing import StandardScaler

# Subset to Advanced_CRC samples
adv_idx = meta_df[meta_df["Stage.3Group"] == "Advanced_CRC"].index
spe_adv = species_df.loc[adv_idx]
mtb_adv = mtb_df.loc[adv_idx]
print(f"Advanced_CRC samples: {len(adv_idx)}")

# Restrict to top 60 species by variance (reduce feature space for GLASSO stability)
top60_sp = spe_adv.var(axis=0).nlargest(60).index
spe_adv  = spe_adv[top60_sp]

# Polyamine columns only
mtb_adv_poly = mtb_adv[[c for c in poly_cols if c in mtb_adv.columns]]
if mtb_adv_poly.empty:
    # Fall back to all metabolites if no polyamine columns matched
    mtb_adv_poly = mtb_adv
    print("WARNING: no polyamine columns found; using all metabolites")

print(f"Features: {spe_adv.shape[1]} species + {mtb_adv_poly.shape[1]} metabolites")

# Concatenate and z-score
joint = pd.concat([spe_adv, mtb_adv_poly], axis=1).dropna()
X     = StandardScaler().fit_transform(joint.values)
n_sp  = spe_adv.shape[1]
n_mt  = mtb_adv_poly.shape[1]

# Condition number check
cond = np.linalg.cond(X.T @ X)
print(f"Condition number of X'X: {cond:.2e}")
if cond > 1e8:
    print("  WARNING: very high condition number — further reducing to top 40 species")
    top40_sp = spe_adv.var(axis=0).nlargest(40).index
    spe_adv  = spe_adv[top40_sp]
    joint    = pd.concat([spe_adv, mtb_adv_poly], axis=1).dropna()
    X        = StandardScaler().fit_transform(joint.values)
    n_sp     = spe_adv.shape[1]
    print(f"  Reduced to {n_sp} species; new condition number: {np.linalg.cond(X.T @ X):.2e}")

# Fit GraphicalLassoCV
gl = GraphicalLassoCV(alphas=[0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5], cv=3, max_iter=1000)
gl.fit(X)
print(f"\nSelected alpha: {gl.alpha_:.4f}")

# Precision matrix → partial correlations
prec = gl.precision_
D    = np.sqrt(np.diag(prec))
D    = np.where(D == 0, 1e-12, D)  # guard against zero diagonal
partial_corr = -prec / np.outer(D, D)
np.fill_diagonal(partial_corr, 1.0)

# Extract species × metabolite cross-block (threshold |pc| >= 0.05)
GLASSO_THRESHOLD = 0.05
sp_names = list(spe_adv.columns)
mt_names = list(mtb_adv_poly.columns)

glasso_rows = []
for i, sp in enumerate(sp_names):
    for j, mt in enumerate(mt_names):
        pc = partial_corr[i, n_sp + j]
        if abs(pc) >= GLASSO_THRESHOLD:
            glasso_rows.append({"Species": sp, "Metabolite": mt,
                                 "PartialCorr": round(pc, 6),
                                 "AbsPartialCorr": round(abs(pc), 6)})

glasso_df = pd.DataFrame(glasso_rows).sort_values("AbsPartialCorr", ascending=False)
glasso_df.to_csv(TABLE_DIR / "d_glasso_partial_correlation_edges_fixed.csv", index=False)
print(f"\nGLASSO edges above threshold {GLASSO_THRESHOLD}: {len(glasso_df)}")
if len(glasso_df):
    print(glasso_df.head(20).to_string(index=False))

Advanced_CRC samples: 163
Features: 60 species + 7 metabolites
Condition number of X'X: 1.09e+03

Selected alpha: 0.0500

GLASSO edges above threshold 0.05: 43
                      Species                Metabolite  PartialCorr  AbsPartialCorr
      Akkermansia muciniphila         C00134_Putrescine    -0.145432        0.145432
             Escherichia coli         C00315_Spermidine    -0.142405        0.142405
      Akkermansia muciniphila                C00062_Arg    -0.141437        0.141437
   Fusobacterium_A mortiferum          C00077_Ornithine     0.123842        0.123842
  Ruminiclostridium_E siraeum          C00077_Ornithine    -0.116801        0.116801
 Barnesiella intestinihominis         C00134_Putrescine    -0.113758        0.113758
 Barnesiella intestinihominis         C00315_Spermidine     0.112499        0.112499
        Faecousia sp003525905         C00134_Putrescine    -0.110293        0.110293
  Sutterella wadsworthensis_A         C00134_Putrescine     0.102895       

## Priority 3 — Genus-Level Enzyme Cross-Linking
Extract genus from species names, look up relevant EC numbers in `module_b3_uniprot_enzymes.csv`,  
and set `E3_kegg = True` wherever the genus harbours a biosynthetic enzyme for the target polyamine.

In [4]:
enzyme_df = pd.read_csv(TABLE_DIR / "module_b3_uniprot_enzymes.csv")
print(f"Enzyme annotations: {len(enzyme_df)} rows")
print("Unique genera:", enzyme_df["Genus"].nunique())
print("Unique EC numbers:", enzyme_df["EC"].nunique())

# Build genus → set(EC) lookup
genus_ec_map = enzyme_df.groupby("Genus")["EC"].apply(set).to_dict()

def has_kegg_enzyme(species: str, polyamine_kegg_id: str) -> bool:
    """Return True if the genus of `species` has a known EC for `polyamine_kegg_id`."""
    genus = extract_genus(species)
    required = POLYAMINE_EC.get(polyamine_kegg_id, [])
    if not required:
        return False
    genus_ecs = genus_ec_map.get(genus.lower(), set())
    return any(ec in genus_ecs for ec in required)

# Verify spot-checks
print("\nSpot-checks (expected True / expected False):")
print(f"  Clostridium sp. + Putrescine (ODC 4.1.1.17): {has_kegg_enzyme('Clostridium leptum', 'C00134')}  [expect True]")
print(f"  Dialister sp. + Putrescine: {has_kegg_enzyme('Dialister sp000434475', 'C00134')}  [expect True – ODC]")
print(f"  Allisonella histaminiformans + Cadaverine (LDC 4.1.1.18): {has_kegg_enzyme('Allisonella histaminiformans', 'C01672')}  [expect False – EC absent]")
print(f"  Acetatifactor intestinalis + Spermidine (2.5.1.16): {has_kegg_enzyme('Acetatifactor intestinalis', 'C00315')}")

Enzyme annotations: 256 rows
Unique genera: 30
Unique EC numbers: 19

Spot-checks (expected True / expected False):
  Clostridium sp. + Putrescine (ODC 4.1.1.17): True  [expect True]
  Dialister sp. + Putrescine: True  [expect True – ODC]
  Allisonella histaminiformans + Cadaverine (LDC 4.1.1.18): False  [expect False – EC absent]
  Acetatifactor intestinalis + Spermidine (2.5.1.16): True


## Priority 4 — Stage-Stratified ML on Advanced CRC
Train Random Forest within Advanced_CRC samples only (Stage_I_II + Stage_III_IV + MP, n≈163)  
to produce stage-independent SHAP feature importances for each polyamine target.

In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
import shap

# Advanced CRC subset
adv_stages = ["Stage_I_II", "Stage_III_IV", "MP"]
adv_idx    = meta_df[meta_df["Stage.6"].isin(adv_stages)].index
print(f"Advanced CRC samples: {len(adv_idx)}")

# Feature matrix: top-variance species (cap at 300 for speed)
top_sp_adv = species_df.loc[adv_idx].var(axis=0).nlargest(300).index
X_adv      = species_df.loc[adv_idx, top_sp_adv].values.astype(float)
sp_names_ml = list(top_sp_adv)
print(f"Feature matrix: {X_adv.shape}")

shap_records = []
cv = KFold(n_splits=10, shuffle=True, random_state=42)

for poly_col in poly_cols:
    kegg_id = poly_col.split("_")[0]
    if poly_col not in mtb_df.columns:
        continue
    y = mtb_df.loc[adv_idx, poly_col].values.astype(float)

    # Skip if target has no variance
    if np.nanstd(y) == 0:
        print(f"  Skipping {poly_col} — zero variance")
        continue

    # Drop NaN rows
    valid  = ~np.isnan(y)
    X_v, y_v = X_adv[valid], y[valid]

    rf = RandomForestRegressor(n_estimators=200, max_features="sqrt",
                               n_jobs=-1, random_state=42)
    scores = cross_val_score(rf, X_v, y_v, cv=cv, scoring="r2")

    # Fit on full advanced-CRC subset for SHAP
    rf.fit(X_v, y_v)
    explainer  = shap.TreeExplainer(rf)
    shap_vals  = explainer.shap_values(X_v)          # (n_samples, n_features)
    mean_shap  = np.abs(shap_vals).mean(axis=0)       # mean |SHAP| per feature

    for sp, sv in zip(sp_names_ml, mean_shap):
        shap_records.append({
            "polyamine":    poly_col,
            "kegg_id":      kegg_id,
            "species":      sp,
            "mean_shap":    float(sv),
            "cv_r2_mean":   float(scores.mean()),
            "cv_r2_std":    float(scores.std()),
            "n_samples":    int(valid.sum()),
        })
    print(f"  {poly_col}: CV R² = {scores.mean():.3f} ± {scores.std():.3f}  |  n={valid.sum()}")

shap_df = (pd.DataFrame(shap_records)
             .sort_values(["polyamine", "mean_shap"], ascending=[True, False])
             .reset_index(drop=True))
shap_df.to_csv(TABLE_DIR / "advanced_crc_ml_shap_results.csv", index=False)
print(f"\nSaved {len(shap_df)} species × polyamine SHAP rows")

Advanced CRC samples: 163
Feature matrix: (163, 300)
  C00062_Arg: CV R² = -0.020 ± 0.243  |  n=163
  C01672_Cadaverine: CV R² = -0.016 ± 0.260  |  n=163
  C00077_Ornithine: CV R² = 0.043 ± 0.168  |  n=163
  C00134_Putrescine: CV R² = 0.163 ± 0.151  |  n=163
  C00315_Spermidine: CV R² = -0.058 ± 0.118  |  n=163
  C00378_Thiamine: CV R² = -0.138 ± 0.225  |  n=163
  C00986_1,3-Diaminopropane: CV R² = 0.024 ± 0.173  |  n=163

Saved 2100 species × polyamine SHAP rows


## Priority 5 — Rebuild Evidence Matrix (7 Streams)
Reconstruct `producer_evidence_matrix_v2.csv` with all 7 evidence columns correctly populated.

| Stream | Source | Criterion |
|---|---|---|
| E1_shap_stable | `advanced_crc_ml_shap_results.csv` | Top-20 SHAP and mean_shap > median + 1 SD |
| E2_spearman | `YACHIDA-CRC-2019_partial_species_polyamine_correlations.csv` | QValue < 0.05 |
| E3_kegg | `module_b3_uniprot_enzymes.csv` + genus mapping | Genus has relevant EC |
| E4_glasso | `d_glasso_partial_correlation_edges_fixed.csv` | AbsPartialCorr ≥ 0.05 |
| E5_mofa | `mofa_top_loadings.csv` | Species in top loadings, co-factor has polyamine \|w\| ≥ 0.20 |
| E6_mediation | `bootstrap_mediation_results.csv` | ACME CI excludes zero |
| E7_within_stage | `within_stage_both_significant.csv` | FDR<0.05 in both Stage_I_II and Stage_III_IV |

In [6]:
# ── Load all evidence sources ─────────────────────────────────────────────────

# Partial correlations (source for E2)
partial_corr_df = pd.read_csv(
    TABLE_DIR / "YACHIDA-CRC-2019_partial_species_polyamine_correlations.csv")
print("partial_corr cols:", list(partial_corr_df.columns))

# SHAP results (source for E1)
shap_df_loaded = pd.read_csv(TABLE_DIR / "advanced_crc_ml_shap_results.csv")

# GLASSO edges (source for E4)
glasso_loaded  = pd.read_csv(TABLE_DIR / "d_glasso_partial_correlation_edges_fixed.csv")
print(f"GLASSO edges: {len(glasso_loaded)}")

# MOFA loadings (source for E5)
mofa_df = pd.read_csv(TABLE_DIR / "mofa_top_loadings.csv")
# Identify which factors have polyamine metabolites with |w| >= 0.20
poly_mofa_factors = set(
    mofa_df[
        (mofa_df["view"] == "metabolite") &
        (mofa_df["name"].isin(POLYAMINE_KEGG)) &
        (mofa_df["abs_weight"] >= 0.20)
    ]["factor"].values
)
print(f"MOFA factors with polyamine |w|≥0.20: {poly_mofa_factors}")
# Species in those factors with |w| >= 0.20
mofa_sp_poly = set(
    mofa_df[
        (mofa_df["factor"].isin(poly_mofa_factors)) &
        (mofa_df["view"] == "species") &
        (mofa_df["abs_weight"] >= 0.20)
    ]["feature"].values
)
print(f"  Species with |w|≥0.20 in polyamine-associated factors: {len(mofa_sp_poly)}")

# Mediation results (source for E6)
med_df = pd.read_csv(TABLE_DIR / "bootstrap_mediation_results.csv")
print("mediation cols:", list(med_df.columns))
# Significant ACME: CI excludes zero
med_sig = med_df[
    (med_df["acme_ci_lo"] * med_df["acme_ci_hi"]) > 0
].copy()
print(f"Significant mediation pairs (CI excludes zero): {len(med_sig)}")
print(med_sig[["species","metabolite_name","acme","acme_ci_lo","acme_ci_hi","q_value"]].to_string(index=False))

# Within-stage both-sig (source for E7)
both_sig_df  = pd.read_csv(TABLE_DIR / "within_stage_both_significant.csv")
either_sig_df = pd.read_csv(TABLE_DIR / "within_stage_either_significant.csv")
print(f"Within-stage either-sig pairs: {len(either_sig_df)}")
print(f"\nWithin-stage both-significant pairs: {len(both_sig_df)}")

partial_corr cols: ['Species', 'Metabolite', 'PartialRho', 'PValue', 'QValue', 'Polyamine_Name']
GLASSO edges: 43
MOFA factors with polyamine |w|≥0.20: {'Factor4'}
  Species with |w|≥0.20 in polyamine-associated factors: 10
mediation cols: ['species', 'metabolite', 'metabolite_name', 'shap_imp', 'pathway', 'acme', 'acme_ci_lo', 'acme_ci_hi', 'a', 'b', 'c_direct', 'c_total', 'prop_mediated', 'p_value', 'n_boot', 'ci', 'q_value']
Significant mediation pairs (CI excludes zero): 8
                      species metabolite_name      acme  acme_ci_lo  acme_ci_hi  q_value
    Acutalibacter sp910580045          C11118  0.016195    0.003753    0.030785      0.0
       Aphodousia sp900553105          C00423 -0.013511   -0.025882   -0.001828      0.0
       Aphodousia sp900553105          C01118 -0.012695   -0.025136   -0.001528      0.0
             Alistipes shahii          C00245 -0.012445   -0.024597   -0.002150      0.0
       Aphodousia sp900553105          C00383 -0.011217   -0.021902   -0.

In [7]:
# ── Build expanded evidence matrix ────────────────────────────────────────────

# Seed from original matrix + GLASSO edges + MOFA poly-factor species
# + mediation pairs + top within-stage pairs
ev_orig = pd.read_csv(TABLE_DIR / "producer_evidence_matrix.csv")

# Normalise original Polyamine column to just the KEGG ID
ev_orig["kegg_id"] = ev_orig["Polyamine"].str.split("_").str[0]

new_rows = []

# ── Add GLASSO edges (species with partial-corr evidence not in original matrix) ──
for _, row in glasso_loaded.iterrows():
    sp  = row["Species"]
    mtb = str(row["Metabolite"])
    kid = mtb.split("_")[0]
    already = ((ev_orig["Species"] == sp) & (ev_orig["kegg_id"] == kid)).any()
    if not already:
        new_rows.append({"Species": sp, "Polyamine": mtb, "kegg_id": kid})
n_glasso_added = len(new_rows)
print(f"GLASSO: {n_glasso_added} new species-metabolite pairs added")

# ── Add MOFA species co-loaded with polyamine metabolites in same factor ──
poly_factors = set(
    mofa_df[
        (mofa_df["view"] == "metabolite") &
        (mofa_df["feature"].str.split("_").str[0].isin(POLYAMINE_KEGG)) &
        (mofa_df["abs_weight"] >= 0.20)
    ]["factor"].values
)
before_mofa = len(new_rows)
for factor in poly_factors:
    factor_poly_mtbs = mofa_df[
        (mofa_df["factor"] == factor) &
        (mofa_df["view"] == "metabolite") &
        (mofa_df["feature"].str.split("_").str[0].isin(POLYAMINE_KEGG)) &
        (mofa_df["abs_weight"] >= 0.20)
    ]["feature"].tolist()
    factor_species = mofa_df[
        (mofa_df["factor"] == factor) &
        (mofa_df["view"] == "species") &
        (mofa_df["abs_weight"] >= 0.20)
    ]["feature"].tolist()
    for sp in factor_species:
        for mtb in factor_poly_mtbs:
            kid = mtb.split("_")[0]
            already = ((ev_orig["Species"] == sp) & (ev_orig["kegg_id"] == kid)).any()
            if not already:
                new_rows.append({"Species": sp, "Polyamine": mtb, "kegg_id": kid})
print(f"MOFA: {len(new_rows) - before_mofa} new pairs added (factors: {poly_factors})")

# ── Add bootstrap mediation candidates not already in the matrix ──
before_med = len(new_rows)
for _, row in med_sig.iterrows():
    sp  = row["species"]
    mtb = row["metabolite"]
    kid = mtb.split("_")[0] if "_" in str(mtb) else str(mtb)
    already = ((ev_orig["Species"] == sp) & (ev_orig["kegg_id"] == kid)).any()
    if not already:
        new_rows.append({"Species": sp, "Polyamine": mtb, "kegg_id": kid})
print(f"Mediation: {len(new_rows) - before_med} new pairs added")

# ── Add top within-stage candidates (polyamine only, same direction) ──
ws_poly = either_sig_df[
    either_sig_df["metabolite"].str.split("_").str[0].isin(POLYAMINE_KEGG) &
    either_sig_df["same_direction"]
].head(20)
before_ws = len(new_rows)
for _, row in ws_poly.iterrows():
    sp  = row["species"]
    mtb = row["metabolite"]
    kid = mtb.split("_")[0]
    already = ((ev_orig["Species"] == sp) & (ev_orig["kegg_id"] == kid)).any()
    if not already:
        new_rows.append({"Species": sp, "Polyamine": mtb, "kegg_id": kid})
print(f"Within-stage: {len(new_rows) - before_ws} new pairs added")

if new_rows:
    ev_new = pd.concat([ev_orig,
                        pd.DataFrame(new_rows)], ignore_index=True)
else:
    ev_new = ev_orig.copy()

# Ensure E1–E7 columns exist (reset to False; we'll re-score everything)
for col in ["E1_shap_stable","E2_spearman","E3_kegg","E4_glasso","E5_mofa","E6_mediation","E7_within_stage"]:
    ev_new[col] = False

print(f"\nEvidence matrix: {len(ev_new)} rows ({len(ev_orig)} original + {len(new_rows)} new)")


GLASSO: 43 new species-metabolite pairs added
MOFA: 10 new pairs added (factors: {'Factor4'})
Mediation: 8 new pairs added
Within-stage: 3 new pairs added

Evidence matrix: 104 rows (40 original + 64 new)


In [8]:
# ── Populate evidence columns ─────────────────────────────────────────────────

# Lookup sets
# E2: species–polyamine partial corr significant at FDR<0.05
partial_sig_set = set(
    zip(partial_corr_df["Species"], partial_corr_df["Metabolite"].str.split("_").str[0]))
# also index by the full metabolite column name for flexibility
partial_sig_fullkey = set(zip(partial_corr_df["Species"], partial_corr_df["Metabolite"]))

# E4: GLASSO edges
if len(glasso_loaded):
    glasso_sig_set = set(zip(glasso_loaded["Species"], glasso_loaded["Metabolite"].str.split("_").str[0]))
else:
    glasso_sig_set = set()

# E6: mediation
med_sig_set = set(zip(med_sig["species"], med_sig["metabolite"].str.split("_").str[0]))

# E7: within-stage both-significant
ws_sig_set = set(zip(either_sig_df["species"], either_sig_df["metabolite"].str.split("_").str[0]))

# E1: SHAP stable — top-20 per polyamine AND mean_shap > median + 1 SD
shap_stable_set = set()
for poly_col, grp in shap_df_loaded.groupby("polyamine"):
    kid   = poly_col.split("_")[0]
    top20 = grp.nlargest(20, "mean_shap")["species"]
    thresh = grp["mean_shap"].median() + grp["mean_shap"].std()
    stable = grp[(grp["species"].isin(top20)) & (grp["mean_shap"] > thresh)]["species"]
    for sp in stable:
        shap_stable_set.add((sp, kid))

# Score each row
def score_row(r):
    sp  = r["Species"]
    kid = r["kegg_id"]
    poly_col_full = r["Polyamine"]   # may be 'C00134_Putrescine' or 'C00134'
    return pd.Series({
        "E1_shap_stable":  (sp, kid) in shap_stable_set,
        "E2_spearman":     (sp, kid) in partial_sig_set,
        "E3_kegg":         has_kegg_enzyme(sp, kid),
        "E4_glasso":       (sp, kid) in glasso_sig_set,
        "E5_mofa":         sp in mofa_sp_poly,
        "E6_mediation":    (sp, kid) in med_sig_set,
        "E7_within_stage": (sp, kid) in ws_sig_set,
    })

ev_scored = ev_new.copy()
ev_scored[["E1_shap_stable","E2_spearman","E3_kegg","E4_glasso","E5_mofa","E6_mediation","E7_within_stage"]] = \
    ev_new.apply(score_row, axis=1)

# Evidence score + confidence tier
EVIDENCE_COLS = ["E1_shap_stable","E2_spearman","E3_kegg","E4_glasso","E5_mofa","E6_mediation","E7_within_stage"]
ev_scored["Evidence_score"] = ev_scored[EVIDENCE_COLS].sum(axis=1)

def confidence_tier(score):
    if score >= 4: return "High (4+)"
    if score >= 3: return "Medium-High (3)"
    if score >= 2: return "Medium (2)"
    if score >= 1: return "Low-Medium (1)"
    return "Insufficient (0)"

ev_scored["Confidence"] = ev_scored["Evidence_score"].map(confidence_tier)

# Save
ev_scored = ev_scored.drop(columns=["kegg_id"], errors="ignore")
ev_scored.to_csv(TABLE_DIR / "producer_evidence_matrix_v2.csv", index=False)
print(f"Saved producer_evidence_matrix_v2.csv ({len(ev_scored)} rows)")

# Summary
print("\nEvidence stream fill rates:")
for col in EVIDENCE_COLS:
    n = ev_scored[col].sum()
    print(f"  {col}: {n} / {len(ev_scored)} ({100*n/len(ev_scored):.0f}%)")

print("\nConfidence tier distribution:")
print(ev_scored["Confidence"].value_counts())

Saved producer_evidence_matrix_v2.csv (104 rows)

Evidence stream fill rates:
  E1_shap_stable: 18 / 104 (17%)
  E2_spearman: 86 / 104 (83%)
  E3_kegg: 7 / 104 (7%)
  E4_glasso: 43 / 104 (41%)
  E5_mofa: 11 / 104 (11%)
  E6_mediation: 8 / 104 (8%)
  E7_within_stage: 5 / 104 (5%)

Confidence tier distribution:
Confidence
Low-Medium (1)     47
Medium (2)         42
Medium-High (3)    13
High (4+)           2
Name: count, dtype: int64


## Summary — Top Candidates

In [9]:
top_candidates = (ev_scored[ev_scored["Evidence_score"] > 0]
                    .sort_values("Evidence_score", ascending=False)
                    .reset_index(drop=True))

print(f"Candidates with Evidence_score > 0: {len(top_candidates)}")
print()
display_cols = ["Species", "Polyamine"] + EVIDENCE_COLS + ["Evidence_score", "Confidence"]
display_cols = [c for c in display_cols if c in top_candidates.columns]
print(top_candidates[display_cols].to_string(index=False))

Candidates with Evidence_score > 0: 104

                         Species                       Polyamine  E1_shap_stable  E2_spearman  E3_kegg  E4_glasso  E5_mofa  E6_mediation  E7_within_stage  Evidence_score      Confidence
         Lachnospira sp000437735               C00315_Spermidine            True         True     True       True    False         False            False               4       High (4+)
         Akkermansia muciniphila                      C00062_Arg            True         True     True       True    False         False            False               4       High (4+)
            Clostridium_A leptum               C00134_Putrescine            True         True     True      False    False         False            False               3 Medium-High (3)
           Faecousia sp003525905               C00134_Putrescine            True         True    False       True    False         False            False               3 Medium-High (3)
         Akkermansia muciniph